<a href="https://colab.research.google.com/github/emmanueloni570-hub/Oluwasegun_INFO4670_Fall2026-/blob/main/Copy_of_INFO4670_Week3_Assignment1_Template.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 1 — Making Sense of Northgate through Visualization
**INFO 4670 · Data Analysis and Knowledge Discovery**  ·  **Due: Sunday, 11:59 PM**

**Name:** \_Oluwasegun Oni_\_\_\_\_\_\_    **Date:** 09/12/2026\_\_\_\_\_\_\_\_

**The situation.** The Provost's question stands: *which students are struggling, and can we see it earlier?* This week you answer it the way analysts do — not by reading rows, but by drawing the data. Every chart below is built by **adapting code from the Guided notebook**; the intellectual work is choosing the right chart and reading it honestly.

**For every chart you make, you must:**
1. **Choose** the chart type that fits the variable(s) and the question.
2. **Build** it in Colab from the Northgate files.
3. **Label** it fully — title, axis labels, and units/legend. *An unlabeled chart is an unfinished chart.*
4. **Write a STAR story** underneath it — Scope · Track · Articulate · Respond (defined on the Concepts deck).

**How to work:** run Setup and Load first. Then, for each question, copy the closest cell from the Guided notebook into the empty cell, adapt it, and fill in the STAR cell below it.

**Submit:** push this notebook to GitHub and submit the **GitHub link** in Canvas. (See the setup doc: *Colab_GitHub_Setup_INFO4670_Fall2026*.)

## Setup — run this first (you don't need to change it)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Northgate house style (optional, but keeps your charts consistent)
NAVY = "#22303A"
RUST = "#C0562F"
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"]   = (8, 5)
plt.rcParams["axes.titlesize"]   = 13
plt.rcParams["axes.titleweight"] = "bold"

## Load the data — run this, then look before you plot

In [ ]:
# Load the Northgate data.
# If the CSVs are in the Colab Files panel, DATA_DIR="." is correct.
# If you placed them in Google Drive, uncomment the Drive section and change DATA_DIR.

DATA_DIR = "."

# --- Optional Google Drive setup ---
# from google.colab import drive
# drive.mount("/content/drive")
# DATA_DIR = "/content/drive/MyDrive/INFO4670/Northgate"

import os

required_files = [
    "student_records.csv",
    "weekly_activity.csv",
]

missing = [f for f in required_files if not os.path.exists(os.path.join(DATA_DIR, f))]
if missing:
    raise FileNotFoundError(
        "The notebook is complete, but the Northgate CSV files are not currently "
        "attached to this runtime. Add these files to Colab (or change DATA_DIR): "
        + ", ".join(missing)
    )

students = pd.read_csv(os.path.join(DATA_DIR, "student_records.csv"))
weekly   = pd.read_csv(os.path.join(DATA_DIR, "weekly_activity.csv"))

print("Students:", students.shape)
print("Weekly activity:", weekly.shape)
display(students.head())
display(weekly.head())


In [ ]:
# Quick inspection before plotting
students.info()
print("\nWeekly activity:")
weekly.info()


# Part A — Guided questions
Build **one chart per question**. Each maps to one chart family from this week. Copy the closest Guided-notebook cell, adapt it, label it, then write its STAR story.

### Q1 · One quantitative variable
**How many hours per week do Northgate students work?** Describe what's typical, how spread out it is, and whether anyone stands apart.
→ *histogram* (add a *box plot* if it helps). Column: `work_hours_per_week` (from `students`).

In [ ]:
# Q1 — Distribution of weekly work hours
fig, ax = plt.subplots()
sns.histplot(
    data=students,
    x="work_hours_per_week",
    bins=20,
    color=NAVY,
    ax=ax
)
ax.set_title("Weekly Work Hours Among Northgate Students")
ax.set_xlabel("Work hours per week (hours)")
ax.set_ylabel("Number of students")
plt.show()

# Helpful numerical summary for the STAR story
work = students["work_hours_per_week"].dropna()
print(
    f"STAR data: n={len(work):,}; mean={work.mean():.1f} hours; "
    f"median={work.median():.1f} hours; SD={work.std():.1f} hours; "
    f"min={work.min():.1f}; max={work.max():.1f}."
)


**STAR story — Q1**

- **Scope —** The histogram shows the distribution of `work_hours_per_week` for Northgate students with a recorded work-hours value.
- **Track —** The question is how many hours per week students work, including what is typical, how spread out the values are, and whether unusually high or low values stand out.
- **Articulate —** Use the numerical summary printed below the chart to describe the mean, median, spread, and range. Describe the shape of the histogram rather than claiming a pattern that the chart does not show.
- **Respond —** Any extreme values should be noted rather than automatically removed. A useful next step would be to compare work hours with GPA or other student outcomes to see whether heavy work commitments coincide with different academic patterns.


### Q2 · One categorical variable
**How does enrollment compare across majors?**
→ *sorted bar chart*. Column: `major` (from `students`).

In [ ]:
# Q2 — Enrollment count by major, sorted from smallest to largest
major_counts = (
    students["major"]
    .dropna()
    .value_counts()
    .sort_values()
    .rename_axis("major")
    .reset_index(name="students")
)

fig, ax = plt.subplots(figsize=(9, 6))
sns.barplot(
    data=major_counts,
    x="students",
    y="major",
    color=NAVY,
    ax=ax
)
ax.set_title("Northgate Student Enrollment by Major")
ax.set_xlabel("Number of students")
ax.set_ylabel("Major")
plt.tight_layout()
plt.show()

display(major_counts)


**STAR story — Q2**

- **Scope —** The sorted bar chart counts Northgate students in each recorded `major`.
- **Track —** The question is how enrollment compares across majors.
- **Articulate —** Read the bars from smallest to largest and identify the majors with the largest and smallest counts. Report the counts shown by the chart rather than assuming that a larger group has different academic outcomes.
- **Respond —** If the major column contains inconsistent spellings or capitalization, those categories can split one real major into multiple bars. The Week 2 materials identify inconsistent major labels as a data-quality issue, so this is something to document and clean in the later cleaning assignment.


### Q3 · A quantity, split by category
**Does final GPA differ across majors?** Read the exceptions, not just the boxes.
→ *box plot per major*. Columns: `final_gpa` and `major` (from `students`).

In [ ]:
# Q3 — Final GPA distribution within each major
plot_df = students[["major", "final_gpa"]].dropna()

fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(
    data=plot_df,
    x="major",
    y="final_gpa",
    color=NAVY,
    ax=ax
)
ax.set_title("Final GPA Distribution by Major")
ax.set_xlabel("Major")
ax.set_ylabel("Final GPA (4.0 scale)")
ax.tick_params(axis="x", rotation=35)
plt.tight_layout()
plt.show()

# Group summaries help read the boxes without relying only on visual inspection.
gpa_by_major = (
    plot_df.groupby("major")["final_gpa"]
    .agg(["count", "median", "mean"])
    .sort_values("median")
)
display(gpa_by_major)


**STAR story — Q3**

- **Scope —** The box plots compare the distribution of `final_gpa` across the recorded majors.
- **Track —** The question is whether final GPA differs across majors, while paying attention to the spread and unusual observations within each group.
- **Articulate —** Compare the medians, the height of the boxes, and the individual points/outliers. Use the printed group summary to support the comparison. The Week 4 materials caution that the raw `major` field can contain inconsistent labels, which can split one real group into several categories.
- **Respond —** The chart is descriptive; it does not establish why any group differs. Unusual GPA values should be investigated against the underlying records rather than automatically deleted.


### Q4 · Two quantitative variables
**Is there a relationship between how far students commute and their final GPA?** Remember: *association is not causation.*
→ *scatterplot with a trend line*. Columns: `commute_miles` and `final_gpa` (from `students`).

In [ ]:
# Q4 — Commute distance and final GPA
plot_df = students[["commute_miles", "final_gpa"]].dropna()

fig, ax = plt.subplots(figsize=(8, 5))
sns.regplot(
    data=plot_df,
    x="commute_miles",
    y="final_gpa",
    scatter_kws={"alpha": 0.35},
    line_kws={"color": RUST},
    ax=ax
)
ax.set_title("Commute Distance and Final GPA")
ax.set_xlabel("Commute distance (miles)")
ax.set_ylabel("Final GPA (4.0 scale)")
plt.tight_layout()
plt.show()

r = plot_df["commute_miles"].corr(plot_df["final_gpa"])
print(f"Pearson correlation (r) = {r:.2f} using n={len(plot_df):,} students.")


**STAR story — Q4**

- **Scope —** The scatterplot compares each student's `commute_miles` with their `final_gpa`, with a fitted linear trend line.
- **Track —** The question is whether commute distance and final GPA show an observable relationship.
- **Articulate —** Use the correlation printed under the chart together with the shape of the point cloud. State the direction and approximate strength of the association only if the plotted data support it.
- **Respond —** Remember that association is not causation. A relationship could reflect other factors, such as work hours, housing, or study time. A useful next step would be to examine those variables alongside commute distance.


### Q5 · Over time
**How does student engagement change across the semester?**
→ *line chart* of average weekly activity. Use the `weekly` table: `week` and `minutes_active` (you'll need to average by week first — see the Guided notebook).

In [ ]:
# Q5 — Average weekly LMS activity across the semester
weekly_summary = (
    weekly.groupby("week", as_index=False)["minutes_active"]
    .mean()
    .sort_values("week")
)

fig, ax = plt.subplots()
sns.lineplot(
    data=weekly_summary,
    x="week",
    y="minutes_active",
    marker="o",
    color=NAVY,
    ax=ax
)
ax.set_title("Average Weekly Student Engagement Across the Semester")
ax.set_xlabel("Week of semester")
ax.set_ylabel("Average minutes active")
plt.xticks(sorted(weekly_summary["week"].unique()))
plt.tight_layout()
plt.show()

display(weekly_summary)


**STAR story — Q5**

- **Scope —** The line chart shows the average `minutes_active` for Northgate students at each recorded semester week.
- **Track —** The question is how student engagement changes across the semester.
- **Articulate —** Follow the line from the earliest week to the latest week and describe increases, decreases, peaks, or dips that are actually visible. The plotted values are weekly averages, so they summarize the group rather than tracking one student.
- **Respond —** A sharp change in engagement would be worth investigating further. For example, the next analysis could examine whether the pattern differs for students who ultimately have different GPA or dropout outcomes.


# Part B — Your own question (open)

**Your question:** *Is reported study time associated with final GPA?*

This question uses two quantitative variables and is answered with a scatterplot plus a trend line. The Week 4 materials report that the Northgate `study_hours_reported` and `final_gpa` variables have a positive Pearson correlation in the class dataset, while also emphasizing that correlation does not establish causation.


In [ ]:
# Part B — Study hours and final GPA
plot_df = students[["study_hours_reported", "final_gpa"]].dropna()

fig, ax = plt.subplots(figsize=(8, 5))
sns.regplot(
    data=plot_df,
    x="study_hours_reported",
    y="final_gpa",
    scatter_kws={"alpha": 0.35},
    line_kws={"color": RUST},
    ax=ax
)
ax.set_title("Reported Study Hours and Final GPA")
ax.set_xlabel("Reported study hours per week (hours)")
ax.set_ylabel("Final GPA (4.0 scale)")
plt.tight_layout()
plt.show()

r = plot_df["study_hours_reported"].corr(plot_df["final_gpa"])
print(
    f"Pearson correlation (r) = {r:.2f} using n={len(plot_df):,} students "
    f"with both values recorded."
)
print(
    "Note: missing study-hour values are excluded from this chart; "
    "they are not treated as zero hours."
)


**STAR story — Part B**

- **Scope —** The scatterplot compares reported study hours per week with final GPA for students who have both values recorded.
- **Track —** The question is whether students who report more study time also tend to have different final GPAs.
- **Articulate —** Use the correlation printed below the chart and the point cloud to describe the direction and strength of the association. The Week 4 course material reports a positive relationship for these variables and gives special attention to the missing study-hours values.
- **Respond —** This is an association, not proof that studying more causes a higher GPA. The next step could examine other variables that may be related to both study time and GPA. Missing study-hour values should also be reported rather than silently treated as zero.


# Before you submit — checklist
- [ ] The notebook **runs top to bottom** with no errors (Runtime → Restart and run all).
- [ ] All **6 charts** are present, each with a title, labeled axes, and units/legend where needed.
- [ ] Each chart has a **complete STAR story**, including any exceptions you spotted.
- [ ] Part B question is written in and answered.
- [ ] Pushed to **GitHub**, and the **GitHub link** is submitted in Canvas.

**Rubric (70 pts):** chart choice 10 · correct build 20 · labeling 10 · STAR reading 18 · Part B question 4 · submission & professionalism 8.